# 01 — Data Cleaning

**Goal:** turn the raw UCI Online Retail transaction log (541,909 rows) into a trustworthy transaction table, ready for RFM aggregation in the next notebook.

**Why this step exists:** RFM math (Recency/Frequency/Monetary) is only as good as the transactions feeding it. Nulls, cancellations, and bad prices/quantities don't crash the pipeline — they silently distort every customer's score. This notebook finds and removes them, with a before/after count at each step so the cleaning is auditable, not a black box.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Load the raw data

In [2]:
df = pd.read_csv("../data/Online_Retail.csv", encoding="latin1", dtype={"CustomerID": "str"})
print("Raw shape:", df.shape)
df.dtypes

Raw shape: (541909, 8)


InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID         str
Country            str
dtype: object

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


## 2. Diagnose the data quality issues

Before fixing anything, measure how big each problem actually is. This is what turns "I cleaned the data" into a specific, defensible claim.

In [4]:
null_customer_pct = df["CustomerID"].isna().mean() * 100
print(f"Rows with null CustomerID: {df['CustomerID'].isna().sum():,} ({null_customer_pct:.1f}%)")

cancelled = df["InvoiceNo"].astype(str).str.startswith("C")
print(f"Cancellation rows (InvoiceNo starts with 'C'): {cancelled.sum():,}")

bad_price = df["UnitPrice"] <= 0
print(f"Rows with UnitPrice <= 0: {bad_price.sum():,}")

bad_qty = df["Quantity"] < 0
print(f"Rows with Quantity < 0: {bad_qty.sum():,}")

Rows with null CustomerID: 135,080 (24.9%)
Cancellation rows (InvoiceNo starts with 'C'): 9,288
Rows with UnitPrice <= 0: 2,517
Rows with Quantity < 0: 10,624


## 3. Apply the fixes, one at a time

Each filter is applied separately with a row-count printed after it — so if the final number ever looks wrong, it's obvious which step caused it.

In [5]:
print(f"Start:                          {len(df):>7,} rows")

# Drop transactions we can't attribute to a customer
df = df.dropna(subset=["CustomerID"])
print(f"After dropping null CustomerID: {len(df):>7,} rows")

# Remove cancelled orders (InvoiceNo starting with 'C')
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
print(f"After removing cancellations:   {len(df):>7,} rows")

# Remove invalid prices (data errors / adjustments, not real revenue)
df = df[df["UnitPrice"] > 0]
print(f"After removing UnitPrice <= 0:  {len(df):>7,} rows")

# Remove negative quantities (returns/adjustments not caught by 'C' prefix)
df = df[df["Quantity"] >= 0]
print(f"After removing Quantity < 0:    {len(df):>7,} rows")

Start:                          541,909 rows
After dropping null CustomerID: 406,829 rows
After removing cancellations:   397,924 rows
After removing UnitPrice <= 0:  397,884 rows
After removing Quantity < 0:    397,884 rows


## 4. Derive Revenue

`Monetary` in the RFM framework needs total spend per line item, not price and quantity separately.

In [6]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df[["Quantity", "UnitPrice", "Revenue"]].describe()

,Quantity,UnitPrice,Revenue
count,397884.000000,397884.000000,397884.000000
mean,12.988238,3.116488,22.397000
std,179.331775,22.097877,309.071041
min,1.000000,0.001000,0.001000
25%,2.000000,1.250000,4.680000
50%,6.000000,1.950000,11.800000
75%,12.000000,3.750000,19.800000
max,80995.000000,8142.750000,168469.600000


## 5. Final check

In [7]:
print(f"Cleaned transactions: {len(df):,}")
print(f"Unique customers:     {df['CustomerID'].nunique():,}")
print(f"Date range:            {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print(f"Total revenue:         £{df['Revenue'].sum():,.2f}")

Cleaned transactions: 397,884
Unique customers:     4,338
Date range:            2010-12-01 to 2011-12-09
Total revenue:         £8,911,407.90


## 6. Save the cleaned table

Saved once here so `02_rfm_analysis.ipynb` (and the SQL step) can load a single trustworthy source instead of re-running this cleaning logic in every notebook.

In [8]:
df.to_csv("../data/cleaned_transactions.csv", index=False)
print("Saved to ../data/cleaned_transactions.csv")

Saved to ../data/cleaned_transactions.csv


## Takeaways

- Started at 541,909 raw rows; four targeted filters (not one big blanket rule) removed nulls, cancellations, and invalid price/quantity rows.
- Each filter was measured *before* being applied — so the cleaning is a documented decision trail, not a guess.
- Output: a clean, transaction-level table with a `Revenue` column, saved for the next step.

**Next up (`02_rfm_analysis.ipynb` + `sql/rfm_aggregation.sql`):** collapse this transaction-level table into one row per customer — Recency, Frequency, Monetary — using both SQL and pandas, and compare that they agree.